# AWS Bedrock Quickstart with Trulens

This notebook demonstrates:
- Installing required packages
- Configuring AWS credentials via environment variables
- Creating a simple RAG-style application
- Adding feedback functions using the Bedrock provider
- Running evaluations and viewing the Trulens dashboard

**Security Note:** Never hardcode credentials. Always use environment variables.

## Setup & Configuration

In [ ]:
# Install required packages
!pip install --quiet trulens-providers-bedrock trulens-sdk langchain faiss-cpu

In [2]:
import os
import sys
from typing import Dict, List, Any

aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_region = os.getenv('AWS_REGION', 'us-west-2')
bedrock_model = os.getenv('BEDROCK_MODEL_ID', 'anthropic.claude-3-sonnet-20240229-v1:0')

print(f'AWS_ACCESS_KEY_ID configured: {bool(aws_access_key)}')
print(f'AWS_SECRET_ACCESS_KEY configured: {bool(aws_secret_key)}')
print(f'AWS_REGION: {aws_region}')
print(f'BEDROCK_MODEL_ID: {bedrock_model}')

if not (aws_access_key and aws_secret_key):
    print('WARNING: AWS credentials not found.')

AWS_ACCESS_KEY_ID configured: False
AWS_SECRET_ACCESS_KEY configured: False
AWS_REGION: us-west-2
BEDROCK_MODEL_ID: anthropic.claude-3-sonnet-20240229-v1:0


## Initialize Bedrock Provider & RAG Components

In [3]:
from trulens.providers.bedrock import Bedrock

provider = Bedrock(region=aws_region, model_id=bedrock_model)
print(f'Bedrock provider initialized with region: {aws_region}')

documents = [
    {'id': 'd1', 'text': 'Trulens is a toolkit for evaluating and giving feedback to LLM-based systems.'},
    {'id': 'd2', 'text': 'Amazon Bedrock provides managed foundation models via an API.'},
    {'id': 'd3', 'text': 'RAG systems combine retrieval with generation for improved accuracy.'}
]
print(f'Loaded {len(documents)} documents for retrieval')

ModuleNotFoundError: No module named 'trulens'

In [ ]:
def simple_retrieve(query: str, docs: List[Dict], top_k: int = 1) -> List[Dict]:
    query_tokens = set(query.lower().split())
    scored = []
    for doc in docs:
        doc_tokens = set(doc['text'].lower().split())
        score = len(query_tokens & doc_tokens)
        scored.append((score, doc))
    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for _, doc in scored[:top_k]]

query = 'What is Trulens and how does it work?'
retrieved_docs = simple_retrieve(query, documents, top_k=2)
context = '\n'.join([f'- {doc["text"]}' for doc in retrieved_docs])
prompt = f'Context:\n{context}\n\nQuestion: {query}\n\nAnswer:'

print(f'Query: {query}')
print(f'Retrieved {len(retrieved_docs)} documents')
print(f'Generated prompt: {prompt[:100]}...')

## Generate Response & Define Feedback Functions

In [ ]:
try:
    response = provider.generate(prompt)
    print(f'Model response: {response}')
except Exception as e:
    print(f'Error calling provider: {e}')
    response = 'Trulens is an evaluation toolkit designed to assess and provide feedback for LLM applications.'

In [ ]:
def relevance_score(query: str, response: str, context: List[Dict] = None) -> Dict[str, Any]:
    query_tokens = set(query.lower().split())
    response_tokens = set(response.lower().split())
    overlap = len(query_tokens & response_tokens)
    total = len(query_tokens | response_tokens)
    score = overlap / total if total > 0 else 0.0
    return {'score': score, 'metric': 'token_overlap'}

def context_relevance(query: str, context: List[Dict] = None) -> Dict[str, Any]:
    if not context:
        return {'score': 0.0, 'metric': 'no_context'}
    query_tokens = set(query.lower().split())
    context_text = ' '.join([doc.get('text', '') for doc in context])
    context_tokens = set(context_text.lower().split())
    overlap = len(query_tokens & context_tokens)
    score = min(overlap / len(query_tokens) if query_tokens else 0.0, 1.0)
    return {'score': score, 'metric': 'context_relevance'}

rel_score = relevance_score(query, response)
ctx_score = context_relevance(query, retrieved_docs)
print(f'Relevance Score: {rel_score["score"]:.3f}')
print(f'Context Relevance: {ctx_score["score"]:.3f}')

## Record & Evaluate Interactions

In [ ]:
from trulens_eval import TruSession

try:
    session = TruSession()
    record = {
        'query': query,
        'response': response,
        'context': retrieved_docs,
        'relevance_score': rel_score,
        'context_relevance': ctx_score
    }
    session.add_record(record)
    print(f'Recorded interaction to TruSession')
    print(f'Query: {query[:50]}...')
    print(f'Metrics: Relevance={rel_score["score"]:.3f}, Context={ctx_score["score"]:.3f}')
except Exception as e:
    print(f'Error creating session: {e}')

## Launch Dashboard

In [ ]:
try:
    from trulens_eval import Dashboard
    dashboard = Dashboard(session=session)
    print('Dashboard created successfully')
    print('To view the dashboard, run: dashboard.run()')
    print('This will start a local server at http://localhost:8501')
except Exception as e:
    print(f'Error initializing dashboard: {e}')

## Execution Steps Summary

**Step 1: Environment Setup**
- Set AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION
- Run cell 1 to install packages

**Step 2: Configuration**
- Run cell 2 to verify AWS configuration

**Step 3: Provider & RAG Initialization**
- Run cell 3 to initialize Bedrock provider
- Run cell 4 to perform document retrieval

**Step 4: Generate Response & Feedback**
- Run cell 5 to generate response from Bedrock model
- Run cell 6 to compute evaluation metrics

**Step 5: Record Interactions**
- Run cell 7 to record interaction with metrics to TruSession

**Step 6: View Results**
- Run cell 8 to initialize dashboard
- Call dashboard.run() to view results at http://localhost:8501

**Notes:**
- All cells include error handling for optional services
- Modify documents and queries for your use case
- Stop dashboard with Ctrl+C or dashboard.stop()